# NullSense — Pothole & Barrier Detection Training (YOLO11, local)

Trains a YOLO11 model on 2 classes — `pothole` and `barriers` — merged from two separate Roboflow exports sitting locally on this machine.

Tuned for a local workstation with **RTX A6000 (48GB VRAM) + 64GB RAM** — larger batch size and a bigger model than the Colab-free-tier notebook, since memory isn't the constraint here.

**Pipeline:** environment check → merge the 2 per-class exports → generate `data.yaml` → train → validate → visualize → export (ONNX / NCNN) for Raspberry Pi edge deployment.


## 1. Environment check

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


In [ ]:
# Install/upgrade Ultralytics (YOLO11 support) if not already present
!pip install -q -U ultralytics onnx onnxruntime onnxsim
import ultralytics
ultralytics.checks()


## 2. Point to your local dataset folders and merge

Expected structure — same as the per-class Roboflow export layout used before, just 2 folders instead of 15:

```
<SOURCE_ROOT>/
├── pothole/
│   ├── data.yaml
│   ├── train/images, train/labels
│   ├── valid/images, valid/labels
│   └── test/images,  test/labels
└── barriers/   (same structure)
```

Edit `SOURCE_ROOT` below to the local path containing both folders.


In [ ]:
import os

CLASS_NAMES = ["barriers", "pothole"]
NAME_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

# EDIT THIS to your local dataset location
SOURCE_ROOT = "/path/to/your/nullsense-dataset"   # <-- folder containing pothole/ and barriers/
MERGED_DIR = "./merged_dataset"

print(f"{len(CLASS_NAMES)} target classes:", CLASS_NAMES)
print("Folders found in source root:", sorted(os.listdir(SOURCE_ROOT)))


In [ ]:
import os, glob, shutil, yaml

def normalize(name: str) -> str:
    """Make folder/class names comparable: lowercase, spaces->underscores."""
    return name.strip().lower().replace(" ", "_").replace("-", "_")

def merge_all_classes():
    NORM_TO_ID = {normalize(k): v for k, v in NAME_TO_ID.items()}

    for split in ["train", "valid", "test"]:
        os.makedirs(f"{MERGED_DIR}/{split}/images", exist_ok=True)
        os.makedirs(f"{MERGED_DIR}/{split}/labels", exist_ok=True)

    summary = []

    class_folders = [f for f in sorted(os.listdir(SOURCE_ROOT))
                      if os.path.isdir(os.path.join(SOURCE_ROOT, f)) and not f.startswith("_")]

    for folder in class_folders:
        folder_path = os.path.join(SOURCE_ROOT, folder)
        local_yaml_path = os.path.join(folder_path, "data.yaml")

        if not os.path.exists(local_yaml_path):
            print(f"[SKIP] {folder}: no data.yaml found")
            continue

        with open(local_yaml_path) as f:
            local_yaml = yaml.safe_load(f)
        local_names = local_yaml.get("names", [])
        if isinstance(local_names, dict):
            local_names = [local_names[i] for i in sorted(local_names)]

        local_id_to_global_id = {}
        for local_id, local_name in enumerate(local_names):
            norm = normalize(local_name)
            if norm in NORM_TO_ID:
                local_id_to_global_id[local_id] = NORM_TO_ID[norm]
            elif normalize(folder) in NORM_TO_ID:
                local_id_to_global_id[local_id] = NORM_TO_ID[normalize(folder)]
            else:
                print(f"[WARN] {folder}: could not match local class '{local_name}' to any global class — skipping this id")

        if not local_id_to_global_id:
            print(f"[SKIP] {folder}: no matchable classes")
            continue

        n_img_copied = 0
        n_lbl_written = 0

        for split in ["train", "valid", "test"]:
            img_dir = os.path.join(folder_path, split, "images")
            lbl_dir = os.path.join(folder_path, split, "labels")
            if not os.path.isdir(img_dir):
                continue

            for img_path in glob.glob(os.path.join(img_dir, "*")):
                fname = os.path.basename(img_path)
                stem, ext = os.path.splitext(fname)
                new_stem = f"{folder}__{stem}"

                dst_img = os.path.join(MERGED_DIR, split, "images", new_stem + ext)
                shutil.copy(img_path, dst_img)
                n_img_copied += 1

                src_lbl = os.path.join(lbl_dir, stem + ".txt")
                dst_lbl = os.path.join(MERGED_DIR, split, "labels", new_stem + ".txt")

                if os.path.exists(src_lbl):
                    out_lines = []
                    with open(src_lbl) as f:
                        for line in f:
                            parts = line.strip().split()
                            if not parts:
                                continue
                            local_id = int(parts[0])
                            if local_id in local_id_to_global_id:
                                new_id = local_id_to_global_id[local_id]
                                out_lines.append(" ".join([str(new_id)] + parts[1:]))
                    with open(dst_lbl, "w") as f:
                        f.write("\n".join(out_lines) + ("\n" if out_lines else ""))
                    n_lbl_written += 1
                else:
                    open(dst_lbl, "w").close()

        summary.append((folder, n_img_copied, n_lbl_written))
        print(f"[OK] {folder}: {n_img_copied} images merged")

    print("\nMerge complete.")
    for folder, n_img, n_lbl in summary:
        print(f"  {folder}: {n_img} images, {n_lbl} label files")

merge_all_classes()


In [ ]:
# Sanity check the merged dataset
from collections import Counter

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(MERGED_DIR, split, "images")
    lbl_dir = os.path.join(MERGED_DIR, split, "labels")
    n_img = len(glob.glob(os.path.join(img_dir, "*")))
    n_lbl = len(glob.glob(os.path.join(lbl_dir, "*.txt")))
    print(f"{split}: {n_img} images, {n_lbl} label files")

class_counts = Counter()
for split in ["train", "valid", "test"]:
    for lbl_file in glob.glob(os.path.join(MERGED_DIR, split, "labels", "*.txt")):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_counts[int(parts[0])] += 1

print("\nAnnotation count per class:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i} {name:10s}: {class_counts.get(i, 0)}")

missing = [CLASS_NAMES[i] for i in range(len(CLASS_NAMES)) if class_counts.get(i, 0) == 0]
if missing:
    print("\n[WARN] These classes have zero annotations — check the matching folder's data.yaml:", missing)


## 3. Generate `data.yaml`

In [ ]:
DATASET_DIR = os.path.abspath(MERGED_DIR)

data_yaml = {
    "path": DATASET_DIR,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

yaml_path = os.path.join(DATASET_DIR, "data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print("Wrote", yaml_path)
print(open(yaml_path).read())


## 4. Train

With 48GB VRAM and 64GB RAM, memory isn't the bottleneck here, so this is tuned differently from the Colab-free-tier notebook:

- **Model:** `yolo11s.pt` (small) by default — a solid accuracy step up from nano, and trivially affordable on this card. Bump to `yolo11m.pt` if you want more accuracy headroom and don't mind slightly longer epochs; both are far below your VRAM ceiling.
- **Batch size:** `64` — comfortable for `yolo11s` at 640px on 48GB. Increase further (96/128) if GPU utilization looks low; decrease if you see any memory warnings.
- **Epochs:** `150` with `patience=30` (early stopping) — a 2-class problem usually converges faster than a many-class one, so this stops automatically once validation mAP plateaus rather than wasting compute running the full 150 regardless.
- **Workers:** matched to a modern multi-core CPU — adjust `workers` down if your CPU has fewer than 12 threads.


In [ ]:
from ultralytics import YOLO

MODEL_SIZE = "yolo11s.pt"   # bump to yolo11m.pt for more accuracy headroom
EPOCHS = 150
IMG_SIZE = 640
BATCH = 64
WORKERS = 12
PROJECT_DIR = "./runs"
RUN_NAME = "nullsense_pothole_barrier_v1"

model = YOLO(MODEL_SIZE)

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=PROJECT_DIR,
    name=RUN_NAME,
    patience=30,
    device=0,
    workers=WORKERS,
    optimizer="auto",
    cos_lr=True,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    plots=True,
)


## 5. Validate

In [ ]:
best_weights = f"{PROJECT_DIR}/{RUN_NAME}/weights/best.pt"
val_model = YOLO(best_weights)
metrics = val_model.val(data=yaml_path, imgsz=IMG_SIZE, split="val")

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Per-class mAP50-95:")
for i, name in enumerate(CLASS_NAMES):
    try:
        print(f"  {name}: {metrics.box.maps[i]:.4f}")
    except IndexError:
        pass


In [ ]:
from IPython.display import Image, display

run_dir = f"{PROJECT_DIR}/{RUN_NAME}"
for fname in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png"]:
    fpath = os.path.join(run_dir, fname)
    if os.path.exists(fpath):
        print(fname)
        display(Image(filename=fpath, width=800))


## 6. Run inference on sample images

In [ ]:
infer_model = YOLO(best_weights)

test_images = glob.glob(f"{DATASET_DIR}/test/images/*")[:5] or glob.glob(f"{DATASET_DIR}/valid/images/*")[:5]

if test_images:
    preds = infer_model.predict(source=test_images, imgsz=IMG_SIZE, conf=0.35, save=True)
    for p in preds:
        print(p.path, "->", len(p.boxes), "detections")
    print("Annotated results saved under runs/detect/predict*/")
else:
    print("No test/valid images found to preview.")


## 7. Export for edge deployment (Raspberry Pi)

- **ONNX** — general-purpose, works with `onnxruntime` on the Pi.
- **NCNN** — typically the fastest option for YOLO on Raspberry Pi CPUs.


In [ ]:
onnx_path = infer_model.export(format="onnx", imgsz=IMG_SIZE, opset=12, simplify=True)
print("Exported ONNX:", onnx_path)


In [ ]:
ncnn_path = infer_model.export(format="ncnn", imgsz=IMG_SIZE)
print("Exported NCNN:", ncnn_path)


---
### Notes
- Since this is a 2-class model, it'll likely converge well before epoch 150 — keep an eye on the `results.png` mAP curve; if it's flat for 20-30 epochs before the run ends, `patience=30` will have already stopped it early, which is expected and fine.
- If GPU utilization (check with `nvidia-smi -l 1` in a terminal while training) stays well under 100%, your bottleneck may be data loading rather than compute — try increasing `WORKERS`, or check that your dataset lives on a fast local disk (not network storage).
- If VRAM usage looks low with `batch=64` on `yolo11s`, you have plenty of headroom to either raise the batch size further or switch to `yolo11m.pt` for better accuracy at roughly the same wall-clock time given this card.
- Once trained, benchmark `best.pt` vs the NCNN export for FPS on the actual Raspberry Pi before finalizing the deployment — training hardware and inference hardware performance don't translate directly.
